In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import yaml
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tueplots import bundles

from bridgeop.utils.trainer import TrainerModule, ScoreModel
from bridgeop.utils.data import DataFactory

2024-10-17 13:20:09.422068: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-17 13:20:09.438151: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-17 13:20:09.442817: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-17 13:20:10.680796: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
plt.rcParams.update(
    bundles.icml2022(column="full", nrows=2, ncols=2, usetex=False, family="sans-serif")
)

### Ellipse

In [6]:
def eval_error(trainer, config, n_eval_pts, n_samples):
    data_config = config.get("data", {})
    data = DataFactory.create(data_config)
    data_evals = data.eval(n_eval_pts)
    x0 = data_evals["x0"]
    v  = data_evals["v"]
    print(f"x0: {x0.shape}, v: {v.shape}")
    
    def true_drift(t, x):
        return (x0 - x) / (t + 1e-8)

    score_model = ScoreModel(x0, trainer)
    db_config = config["diffusion_bridge"]
    db_config["sde_kwargs"]["W_shape"] = [n_eval_pts[0], 2]
    trainer.update_db(db_config)
    reverse_path = trainer.db.solve_reverse_bridge(
        rng_key=jax.random.PRNGKey(config["training"]["seed"]),
        xT=v,
        model=score_model,
        n_batches=n_samples,
        return_drift=True
    )
    est_drifts = reverse_path.fs.reshape((n_samples, -1, *n_eval_pts, 2))
    true_drifts = jax.vmap(
        jax.vmap(
            true_drift,
            in_axes=(0, 0),
            out_axes=0
        ),
        in_axes=(None, 0),
        out_axes=0
        )(reverse_path.ts[1:], reverse_path.xs[:, 1:])
    drift_error = jnp.linalg.norm(est_drifts - true_drifts, axis=-1)
    
    end_shape = reverse_path.xs[:, -1, ...]
    end_shape_error = jnp.linalg.norm(end_shape - v[None, ...], axis=-1)
    return jnp.mean(drift_error), jnp.mean(end_shape_error)

In [9]:
n_eval_pts = [32, 64, 128, 256]
drift_errors8 = []
end_shape_errors8 = []
ckpt_path = f"../ckpts/ellipse_brownian_neuralop_864_fourier_modes_64pts/"
config_path = "../src/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = config["ellipse_brownian"]
    
cwd = os.getcwd()
ckpt_path = os.path.join(cwd, ckpt_path)
ckpt_path = os.path.normpath(ckpt_path)
config["training"]["dir"] = ckpt_path
config["neural_op"]["n_modes_per_layer"] = [[8, ], [6, ], [4, ]]

trainer = TrainerModule(config)
trainer.train_model(x0=None, mode="pretrained", step=None)
for n_eval_pt in n_eval_pts:
    drift_error, end_shape_error = eval_error(trainer, config, n_eval_pts=(n_eval_pt, ), n_samples=64)
    drift_errors8.append(drift_error)
    end_shape_errors8.append(end_shape_error)

Number of trainable parameters: 131,314


/home/gefan/miniconda3/envs/bridgeop/lib/python3.12/site-packages/orbax/checkpoint/type_handlers.py:1330: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Model loaded from /home/gefan/projects/scoreoperator/ckpts/ellipse_brownian_neuralop_864_fourier_modes_64pts/checkpoint__step_10000
x0: (32, 2), v: (32, 2)
x0: (64, 2), v: (64, 2)
x0: (128, 2), v: (128, 2)
x0: (256, 2), v: (256, 2)


In [10]:
print(list(drift_errors8))


[Array(1.5966331, dtype=float32), Array(1.5959401, dtype=float32), Array(1.5963604, dtype=float32), Array(1.5983871, dtype=float32)]


In [11]:
n_eval_pts = [32, 64, 128, 256]
drift_errors16 = []
end_shape_errors16 = []
ckpt_path = f"../ckpts/ellipse_brownian_neuralop_16128_fourier_modes_64pts/"
config_path = "../src/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = config["ellipse_brownian"]
    
cwd = os.getcwd()
ckpt_path = os.path.join(cwd, ckpt_path)
ckpt_path = os.path.normpath(ckpt_path)
config["training"]["dir"] = ckpt_path
config["neural_op"]["n_modes_per_layer"] = [[16, ], [12, ], [8, ]]

trainer = TrainerModule(config)
trainer.train_model(x0=None, mode="pretrained", step=None)
for n_eval_pt in n_eval_pts:
    drift_error, end_shape_error = eval_error(trainer, config, n_eval_pts=(n_eval_pt, ), n_samples=64)
    drift_errors16.append(drift_error)
    end_shape_errors16.append(end_shape_error)
print(list(drift_errors16))
print(list(end_shape_errors16))


Number of trainable parameters: 185,586


/home/gefan/miniconda3/envs/bridgeop/lib/python3.12/site-packages/orbax/checkpoint/type_handlers.py:1330: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Model loaded from /home/gefan/projects/scoreoperator/ckpts/ellipse_brownian_neuralop_16128_fourier_modes_64pts/checkpoint__step_10000
x0: (32, 2), v: (32, 2)
x0: (64, 2), v: (64, 2)
x0: (128, 2), v: (128, 2)
x0: (256, 2), v: (256, 2)
[Array(1.5894046, dtype=float32), Array(1.5815413, dtype=float32), Array(1.5779274, dtype=float32), Array(1.5768439, dtype=float32)]
[Array(0.10024854, dtype=float32), Array(0.11251711, dtype=float32), Array(0.12101574, dtype=float32), Array(0.12271306, dtype=float32)]


In [12]:
n_eval_pts = [32, 64, 128, 256]
drift_errors32 = []
end_shape_errors32 = []
ckpt_path = f"../ckpts/ellipse_brownian_neuralop_322416_fourier_modes_64pts/"
config_path = "../src/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = config["ellipse_brownian"]
    
cwd = os.getcwd()
ckpt_path = os.path.join(cwd, ckpt_path)
ckpt_path = os.path.normpath(ckpt_path)
config["training"]["dir"] = ckpt_path
config["neural_op"]["n_modes_per_layer"] = [[32, ], [24, ], [16, ]]

trainer = TrainerModule(config)
trainer.train_model(x0=None, mode="pretrained", step=None)
for n_eval_pt in n_eval_pts:
    drift_error, end_shape_error = eval_error(trainer, config, n_eval_pts=(n_eval_pt, ), n_samples=64)
    drift_errors32.append(drift_error)
    end_shape_errors32.append(end_shape_error)
print(list(drift_errors32))
print(list(end_shape_errors16))

Number of trainable parameters: 294,130


/home/gefan/miniconda3/envs/bridgeop/lib/python3.12/site-packages/orbax/checkpoint/type_handlers.py:1330: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Model loaded from /home/gefan/projects/scoreoperator/ckpts/ellipse_brownian_neuralop_322416_fourier_modes_64pts/checkpoint__step_10000
x0: (32, 2), v: (32, 2)
x0: (64, 2), v: (64, 2)
x0: (128, 2), v: (128, 2)
x0: (256, 2), v: (256, 2)
[Array(1.538905, dtype=float32), Array(1.527128, dtype=float32), Array(1.5198059, dtype=float32), Array(1.5142002, dtype=float32)]
[Array(0.10024854, dtype=float32), Array(0.11251711, dtype=float32), Array(0.12101574, dtype=float32), Array(0.12271306, dtype=float32)]
